# Single-Cell Clustering Analysis

This notebook performs clustering and cell type annotation:
- Leiden/Louvain clustering
- UMAP visualization
- Marker gene identification
- Cell type annotation (manual/automated)

**Input**: AnnData object (.h5ad) after QC
**Outputs**: Clustered UMAP, marker genes, annotations

In [ ]:
# Parameters
RESOLUTION = 0.5  # Clustering resolution
N_NEIGHBORS = 15  # For neighborhood graph
N_PCS = 30  # Principal components for clustering
MIN_CELLS = 3  # Minimum cells per cluster
TOP_MARKERS = 10  # Top markers per cluster
FIGURE_DPI = 150

In [ ]:
import scanpy as sc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from galaxy_notebook import get, put, list_inputs

sc.settings.verbosity = 2
sc.settings.set_figure_params(dpi=FIGURE_DPI, frameon=False)
plt.rcParams['figure.dpi'] = FIGURE_DPI

In [ ]:
# List available inputs
print("Available inputs:")
for inp in list_inputs():
    print(f"  [{inp['index']}] {inp['name']}")

In [ ]:
# Load AnnData
input_path = get(0, format='path')
adata = sc.read_h5ad(input_path)
print(f"Loaded: {adata.n_obs} cells x {adata.n_vars} genes")
adata

## 1. Preprocessing (if needed)

In [ ]:
# Check if already processed
if 'X_pca' not in adata.obsm:
    print("Running preprocessing...")
    
    # Normalize
    sc.pp.normalize_total(adata, target_sum=1e4)
    sc.pp.log1p(adata)
    
    # Find highly variable genes
    sc.pp.highly_variable_genes(adata, n_top_genes=2000)
    adata.raw = adata
    adata = adata[:, adata.var.highly_variable]
    
    # Scale
    sc.pp.scale(adata, max_value=10)
    
    # PCA
    sc.tl.pca(adata, n_comps=50)
    
    print("Preprocessing complete")
else:
    print("Data already preprocessed")

## 2. Neighborhood Graph and Clustering

In [ ]:
# Compute neighborhood graph
sc.pp.neighbors(adata, n_neighbors=N_NEIGHBORS, n_pcs=N_PCS)
print(f"Computed neighbors with k={N_NEIGHBORS}, n_pcs={N_PCS}")

In [ ]:
# Run Leiden clustering
sc.tl.leiden(adata, resolution=RESOLUTION, key_added='leiden')

# Also run at different resolutions for comparison
for res in [0.3, 0.8, 1.0]:
    sc.tl.leiden(adata, resolution=res, key_added=f'leiden_{res}')

n_clusters = adata.obs['leiden'].nunique()
print(f"Found {n_clusters} clusters at resolution {RESOLUTION}")

In [ ]:
# Compute UMAP
if 'X_umap' not in adata.obsm:
    sc.tl.umap(adata)
    print("UMAP computed")

In [ ]:
# Plot UMAP with clusters
fig, axes = plt.subplots(2, 2, figsize=(14, 14))

# Main clustering
sc.pl.umap(adata, color='leiden', ax=axes[0, 0], show=False, 
           title=f'Leiden Clustering (res={RESOLUTION})', legend_loc='on data')

# Different resolutions
sc.pl.umap(adata, color='leiden_0.3', ax=axes[0, 1], show=False, 
           title='Leiden (res=0.3)', legend_loc='on data')
sc.pl.umap(adata, color='leiden_0.8', ax=axes[1, 0], show=False, 
           title='Leiden (res=0.8)', legend_loc='on data')
sc.pl.umap(adata, color='leiden_1.0', ax=axes[1, 1], show=False, 
           title='Leiden (res=1.0)', legend_loc='on data')

plt.tight_layout()
put(fig, 'clustering_umap.png')
plt.show()

## 3. Marker Gene Identification

In [ ]:
# Find marker genes
sc.tl.rank_genes_groups(adata, 'leiden', method='wilcoxon')

# Get marker gene table
markers = sc.get.rank_genes_groups_df(adata, group=None)
print(f"Found markers for {markers['group'].nunique()} clusters")
markers.head(20)

In [ ]:
# Plot marker gene heatmap
top_markers_per_cluster = []
for cluster in adata.obs['leiden'].unique():
    cluster_markers = markers[markers['group'] == cluster].head(TOP_MARKERS)['names'].tolist()
    top_markers_per_cluster.extend(cluster_markers)

# Remove duplicates while preserving order
seen = set()
unique_markers = [x for x in top_markers_per_cluster if not (x in seen or seen.add(x))]

fig = sc.pl.rank_genes_groups_heatmap(
    adata, n_genes=TOP_MARKERS, show=False, 
    groupby='leiden', cmap='viridis'
)
put(plt.gcf(), 'marker_heatmap.png')
plt.show()

In [ ]:
# Dot plot of top markers
# Get top 3 markers per cluster
top3_per_cluster = markers.groupby('group').head(3)['names'].unique().tolist()

fig = sc.pl.dotplot(adata, top3_per_cluster[:30], groupby='leiden', 
                    dendrogram=True, show=False)
put(plt.gcf(), 'marker_dotplot.png')
plt.show()

## 4. Cluster Statistics

In [ ]:
# Cluster size distribution
cluster_counts = adata.obs['leiden'].value_counts().sort_index()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar plot
cluster_counts.plot(kind='bar', ax=axes[0], color='#3498db', edgecolor='white')
axes[0].set_xlabel('Cluster')
axes[0].set_ylabel('Number of Cells')
axes[0].set_title('Cells per Cluster')

# Pie chart
colors = plt.cm.tab20(np.linspace(0, 1, len(cluster_counts)))
axes[1].pie(cluster_counts.values, labels=cluster_counts.index, 
            autopct='%1.1f%%', colors=colors)
axes[1].set_title('Cluster Proportions')

plt.tight_layout()
put(fig, 'cluster_statistics.png')
plt.show()

In [ ]:
# Calculate cluster statistics
cluster_stats = pd.DataFrame({
    'n_cells': cluster_counts,
    'percentage': cluster_counts / cluster_counts.sum() * 100
})

# Add mean genes per cell
if 'n_genes_by_counts' in adata.obs.columns:
    cluster_stats['mean_genes'] = adata.obs.groupby('leiden')['n_genes_by_counts'].mean()
    cluster_stats['mean_counts'] = adata.obs.groupby('leiden')['total_counts'].mean()

print("Cluster Statistics:")
cluster_stats

## 5. Common Marker Visualization

In [ ]:
# Common cell type markers (if present in data)
common_markers = {
    'T cells': ['CD3D', 'CD3E', 'CD4', 'CD8A'],
    'B cells': ['CD19', 'MS4A1', 'CD79A'],
    'NK cells': ['NCAM1', 'NKG7', 'GNLY'],
    'Monocytes': ['CD14', 'LYZ', 'S100A9'],
    'Dendritic': ['FCER1A', 'CD1C'],
    'Epithelial': ['EPCAM', 'KRT19'],
    'Fibroblast': ['COL1A1', 'DCN'],
    'Endothelial': ['PECAM1', 'VWF']
}

# Check which markers exist in data
gene_names = adata.raw.var_names if adata.raw else adata.var_names
available_markers = {}
for cell_type, markers_list in common_markers.items():
    existing = [m for m in markers_list if m in gene_names]
    if existing:
        available_markers[cell_type] = existing

print(f"Available marker genes for {len(available_markers)} cell types")
for ct, markers_list in available_markers.items():
    print(f"  {ct}: {markers_list}")

In [ ]:
# Plot common markers on UMAP
if available_markers:
    all_markers = [m for markers_list in available_markers.values() for m in markers_list]
    
    # Plot up to 12 markers
    plot_markers = all_markers[:12]
    
    fig = sc.pl.umap(adata, color=plot_markers, ncols=4, show=False,
                    use_raw=True if adata.raw else False)
    put(plt.gcf(), 'marker_umap.png')
    plt.show()

## 6. Summary

In [ ]:
# Generate summary
summary = f"""
# Single-Cell Clustering Analysis Summary

## Dataset
- Total cells: {adata.n_obs:,}
- Total genes: {adata.n_vars:,}

## Clustering Parameters
- Algorithm: Leiden
- Resolution: {RESOLUTION}
- Number of neighbors: {N_NEIGHBORS}
- Principal components: {N_PCS}

## Results
- Number of clusters: {n_clusters}
- Largest cluster: {cluster_counts.max():,} cells ({cluster_counts.idxmax()})
- Smallest cluster: {cluster_counts.min():,} cells ({cluster_counts.idxmin()})

## Top Markers by Cluster
"""

for cluster in sorted(adata.obs['leiden'].unique(), key=int):
    cluster_markers = markers[markers['group'] == cluster].head(5)['names'].tolist()
    summary += f"\n**Cluster {cluster}** ({cluster_counts[cluster]:,} cells):\n"
    summary += f"  {', '.join(cluster_markers)}\n"

print(summary)
put(summary, 'clustering_summary.md')

In [ ]:
# Save outputs
# Save updated AnnData
adata.write('/export/clustered_data.h5ad')
put(adata, 'clustered_data.h5ad')

# Save marker genes
put(markers, 'marker_genes.tsv')

# Save cluster assignments
cluster_assignments = adata.obs[['leiden']].copy()
put(cluster_assignments.reset_index(), 'cluster_assignments.tsv')

# Save cluster statistics
put(cluster_stats.reset_index(), 'cluster_statistics.tsv')

print("\n=== Analysis Complete ===")
print("Generated outputs:")
print("  - clustering_umap.png")
print("  - marker_heatmap.png")
print("  - marker_dotplot.png")
print("  - cluster_statistics.png")
print("  - marker_umap.png")
print("  - clustering_summary.md")
print("  - clustered_data.h5ad")
print("  - marker_genes.tsv")
print("  - cluster_assignments.tsv")
print("  - cluster_statistics.tsv")